# Setup

In [ ]:
%load_ext autoreload
%autoreload 2

# %matplotlib widget

In [ ]:
from rich import print
from dunetrg.data.workspace import TriggerActivityWorkspace

import mplhep as hep
import matplotlib.pyplot as plt

# Data

In [ ]:
import os
from pathlib import Path

# TODO(migration): stray path, not under TPV_DATA_ROOT/vd/...; found at
# tpvalidator/data/vd/1x8x14/switft_tas_old/radbkg_5000.root in the source repo.
# taws = TriggerActivityWorkspace('../../../radbkg_5000.root', 'taFinder')
taws = TriggerActivityWorkspace(str(Path(os.environ['TPV_DATA_ROOT']) / 'vd/1x8x14/swift_tafinding/ta-finder_vd-1x8x14-radbkg.root'), 'taFinder')


# Analyzer


In [ ]:
from dunetrg.analysis.base import TrgWorkspaceAnalyzer
class SwiftTABackgroundsAnalyzer(TrgWorkspaceAnalyzer):
    ...


    

In [ ]:
sa = SwiftTABackgroundsAnalyzer(taws)

# Analysis

In [ ]:
taws.ta_win_cluster_stats.sadc

In [ ]:
simulated_readout_time_s = len(taws.event_summary)*8500*0.5e-6
num_crps = 28
num_crps_fdvd = 160

print(f"Simulated time in seconds {simulated_readout_time_s}")

crtp_time = simulated_readout_time_s*num_crps
print(f"Simulated crtp_time time in seconds {crtp_time}")

fdvd_time = simulated_readout_time_s*num_crps/num_crps_fdvd
print(f"Simulated fdvd10k time in seconds {fdvd_time}")




In [ ]:
taws.ta_clusters

In [ ]:
from dunetrg.analysis.histograms import make_regaxis, build_histogram, cumsum_hist_nd

clusters = taws.ta_win_cluster_stats
cl_sadc_axis = make_regaxis(clusters, 'max_cluster_sadc', 200)


h = build_histogram(clusters, [
    cl_sadc_axis
])


# Plot the sum adc of the top cluster in all inspect windows
fig, axes = plt.subplots(2, 2, figsize=(14,10))
ax = axes[0][0]
hep.histplot(h[1:], ax=ax)
ax.set_title('Distribution of sadc of top cluster')
ax.set_ylabel('counts')
ax.set_yscale('log')

# Plot the right-to-left cumulative sum
ax = axes[0][1]
ch = cumsum_hist_nd(h[1:], 0, 'right')
hep.histplot(ch, ax=ax)
ax.set_title('Cumulative sum')
ax.set_yscale('log')


ax = axes[1][0]

rh = (
    # Cumuative sum histogram
    ch / fdvd_time
)

hep.histplot(rh, ax=ax)
ax.set_title('TA inspect-accept rate')
ax.set_yscale('log')
ax.set_ylabel('Rate [Hz]')
ax.set_xlabel('cluster_sadc_min')

ax.axhline(1, label='1 Hz', c='red', ls='-.')
ax.legend()



target_rate = 1.

nb = len(rh.axes[0])

for i in range(nb-1, 0, -1):
    if rh[i].value > target_rate:
        print(rh[i], rh.axes[0].centers[i])
        print(rh[i+1], rh.axes[0].centers[i+1])

        thres_left = rh.axes[0].centers[i]
        thres_right = rh.axes[0].centers[i+1]
        break


ax = axes[1][1]

ax.text(0.5, 0.5, f'1 Hz threshold ({thres_left}-{thres_right})')

In [ ]:
taws.tree_names 


In [ ]:
taws.ta_win_stats.sadc.hist(bins=100)